# LST Geospatial Analysis: Miami vs Phoenix

This notebook demonstrates the LST pipeline comparing two cities in different climate zones:
- **Miami**: Humid Subtropical climate (coastal, moderate temperatures)
- **Phoenix**: Hot Desert climate (inland, extreme temperatures)

We'll analyze Land Surface Temperature data from MODIS MOD11A1 (2016-2026) using:
- **Moran's I**: Spatial autocorrelation statistic
- **Temporal trends**: Changes across seasons and years
- **Hotspot analysis**: Identification of high-temperature areas

## Setup & Imports

In [ ]:
!git clone -b lst-htIndex-compare https://github.com/cchen744/uhi-extreme-heat-response.git -q

In [1]:
!pip install --upgrade earthengine-api -q
import ee
print(dir(ee))  # See what's available

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.9/481.9 kB 2.6 MB/s eta 0:00:00
['Algorithms', 'Any', 'ApiFunction', 'Array', 'Authenticate', 'Blob', 'Classifier', 'Clusterer', 'Collection', 'ComputedObject', 'ConfusionMatrix', 'CustomFunction', 'Date', 'DateRange', 'Dictionary', 'EEException', 'Element', 'Encodable', 'ErrorMargin', 'Feature', 'FeatureCollection', 'Filter', 'Function', 'Geometry', 'Hashable', 'Image', 'ImageCollection', 'Initialize', 'Join', 'Kernel', 'List', 'Model', 'NO_PROJECT_EXCEPTION', 'Number', 'PixelType', 'Projection', 'Reducer', 'Reset', 'Sequence', 'Serializer', 'ServiceAccountCredentials', 'String', 'TYPE_CHECKING', 'Terrain', '_AlgorithmsContainer', '_DYNAMIC_CLASSES', '_HAS_DYNAMIC_ATTRIBUTES', '_InitializeDeprecatedAssets', '_InitializeGeneratedClasses', '_InitializeUnboundMethods', '_MakeClass', '_Promote', '_ResetGeneratedClasses', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__

In [5]:
# Cell 1: Auth first
ee.Authenticate()
ee.Initialize(project='extremeweatheruhi')
print("✓ GEE ready")

✓ GEE ready


In [6]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("✓ Imports successful")

✓ Imports successful


In [24]:
# Clone specific branch

import importlib
sys.path.insert(0, '/content/uhi-extreme-heat-response/lst-htIndex-compare/')

import importlib
import sys

# Remove old module
if 'lst_pipeline' in sys.modules:
    del sys.modules['lst_pipeline']

# Re-import
from lst_pipeline import (
    create_config,
    LSSTPipeline,
    CITY_PRESETS,
    SpatialAnalyzer,
    LSTVisualizer
)
print("✓ Imported")

✓ Imported


## Google Earth Engine Authentication

**Note**: Run On first run, this will open a browser for authentication. Run the command that appears in the output.

## Run Pipeline: Miami

In [25]:
import ee

config = create_config('Miami', start_year=2020, end_year=2020)
geom = config.create_geom()

# Test basic MODIS fetch
collection = ee.ImageCollection('MODIS/061/MOD11A1').filterDate('2020-06-01', '2020-09-01').filterBounds(geom)
print(f"Images found: {collection.size().getInfo()}")

# Get one image
img = collection.first()
print(f"Image: {img.getInfo()}")

Images found: 92
Image: {'type': 'Image', 'bands': [{'id': 'LST_Day_1km', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [43200, 21600], 'crs': 'SR-ORG:6974', 'crs_transform': [926.6254331383326, 0, -20015109.355797, 0, -926.6254331391667, 10007554.677903]}, {'id': 'QC_Day', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [43200, 21600], 'crs': 'SR-ORG:6974', 'crs_transform': [926.6254331383326, 0, -20015109.355797, 0, -926.6254331391667, 10007554.677903]}, {'id': 'Day_view_time', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [43200, 21600], 'crs': 'SR-ORG:6974', 'crs_transform': [926.6254331383326, 0, -20015109.355797, 0, -926.6254331391667, 10007554.677903]}, {'id': 'Day_view_angle', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [43200, 21600], 'crs': 'SR-ORG:6974', 'crs_transform': [926.6254331383326, 0

In [26]:
# Use preset Miami bounds instead of FIPS
miami_config = create_config('Miami', start_year=2020, end_year=2024)

print(f"City: {miami_config.city_name}")
print(f"Bounds: {miami_config.city_bounds}")
print(f"MODIS Product: {miami_config.modis_product}")

City: Miami
Bounds: [-80.3, 25.7, -80.1, 25.9]
MODIS Product: MODIS/061/MOD11A1


In [27]:
# Run pipeline for Miami
miami_pipeline = LSSTPipeline(miami_config)
miami_results = miami_pipeline.run()

GEE not initialized. Run ee.Authenticate() first if needed.

LST Pipeline: Miami
Period: 2020-2024

Step 1: Fetching MODIS data...
  2020 DJF month 12: 31 images
  2020 DJF month 1: 31 images
  2020 DJF month 2: 29 images
  2020 DJF merged: 91 images
  2020 MAM month 3: 31 images
  2020 MAM month 4: 30 images
  2020 MAM month 5: 31 images
  2020 MAM merged: 92 images
  2020 JJA month 6: 30 images
  2020 JJA month 7: 31 images
  2020 JJA month 8: 31 images
  2020 JJA merged: 92 images
  2020 SON month 9: 30 images
  2020 SON month 10: 31 images
  2020 SON month 11: 30 images
  2020 SON merged: 91 images
  2021 DJF month 12: 31 images
  2021 DJF month 1: 31 images
  2021 DJF month 2: 28 images
  2021 DJF merged: 90 images
  2021 MAM month 3: 31 images
  2021 MAM month 4: 30 images
  2021 MAM month 5: 31 images
  2021 MAM merged: 92 images
  2021 JJA month 6: 30 images
  2021 JJA month 7: 31 images
  2021 JJA month 8: 31 images
  2021 JJA merged: 92 images
  2021 SON month 9: 30 images
  

In [28]:
# Display Miami statistics
miami_stats = miami_results['statistics']
print("\n" + "="*60)
print("MIAMI - Statistical Summary")
print("="*60)
print(f"\nSeasonal means:")
print(miami_stats.groupby('season')[['mean_temp', 'morans_i']].mean().round(2))
print(f"\nYearly means:")
print(miami_stats.groupby('year')[['mean_temp', 'morans_i']].mean().round(2))
print(f"\nFirst few rows:")
print(miami_stats.head(10))


MIAMI - Statistical Summary

Seasonal means:


KeyError: 'season'

## Run Pipeline: Phoenix

In [ ]:
# Create config for Phoenix
phoenix_config = create_config('Phoenix', start_year=2020, end_year=2024, output_dir='./outputs')
print(f"City: {phoenix_config.city_name}")
print(f"Bounds: {phoenix_config.city_bounds}")
print(f"MODIS Product: {phoenix_config.modis_product}")
print(f"Output Directory: {phoenix_config.output_dir}")

In [ ]:
# Run pipeline for Phoenix
phoenix_pipeline = LSSTPipeline(phoenix_config)
phoenix_results = phoenix_pipeline.run()

In [ ]:
# Display Phoenix statistics
phoenix_stats = phoenix_results['statistics']
print("\n" + "="*60)
print("PHOENIX - Statistical Summary")
print("="*60)
print(f"\nSeasonal means:")
print(phoenix_stats.groupby('season')[['mean_temp', 'morans_i']].mean().round(2))
print(f"\nYearly means:")
print(phoenix_stats.groupby('year')[['mean_temp', 'morans_i']].mean().round(2))
print(f"\nFirst few rows:")
print(phoenix_stats.head(10))

## Comparative Analysis

### Temperature Comparison

In [ ]:
# Create comparison dataframe
miami_stats['city'] = 'Miami'
phoenix_stats['city'] = 'Phoenix'
combined = pd.concat([miami_stats, phoenix_stats], ignore_index=True)

# Summary by city
print("\n" + "="*60)
print("OVERALL COMPARISON: Miami vs Phoenix")
print("="*60)
comparison = combined.groupby('city')[[
    'mean_temp', 'median_temp', 'std_temp', 'min_temp', 'max_temp',
    'morans_i', 'morans_i_pvalue', 'hotspot_pct'
]].agg(['mean', 'std']).round(2)
print(comparison)

In [ ]:
# Plot: Temperature trends by season
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

seasons = ['DJF', 'MAM', 'JJA', 'SON']

for idx, season in enumerate(seasons):
    ax = axes[idx // 2, idx % 2]

    miami_season = miami_stats[miami_stats['season'] == season].groupby('year')['mean_temp'].mean()
    phoenix_season = phoenix_stats[phoenix_stats['season'] == season].groupby('year')['mean_temp'].mean()

    ax.plot(miami_season.index, miami_season.values, marker='o', label='Miami', linewidth=2)
    ax.plot(phoenix_season.index, phoenix_season.values, marker='s', label='Phoenix', linewidth=2)

    ax.set_title(f'{season} - Mean LST Trend', fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Mean LST (°C)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./outputs/temperature_trends_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: temperature_trends_comparison.png")

### Moran's I Analysis (Spatial Autocorrelation)

In [ ]:
# Moran's I interpretation
print("\nMoran's I Interpretation:")
print("  > 0: Spatial clustering (similar values clustered together)")
print("  = 0: Random spatial distribution")
print("  < 0: Spatial dispersion (opposite values near each other)")
print("\nHigher Moran's I = stronger spatial autocorrelation")

print("\n" + "="*60)
print("MORAN'S I by City")
print("="*60)

miami_morans = miami_stats.groupby('season')['morans_i'].mean()
phoenix_morans = phoenix_stats.groupby('season')['morans_i'].mean()

print("\nMiami (Moran's I by season):")
print(miami_morans.round(3))
print(f"Mean: {miami_morans.mean():.3f}")

print("\nPhoenix (Moran's I by season):")
print(phoenix_morans.round(3))
print(f"Mean: {phoenix_morans.mean():.3f}")

In [ ]:
# Plot: Moran's I temporal evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By season
miami_morans_by_season = miami_stats.groupby('season')['morans_i'].mean()
phoenix_morans_by_season = phoenix_stats.groupby('season')['morans_i'].mean()

x = np.arange(len(miami_morans_by_season))
width = 0.35

axes[0].bar(x - width/2, miami_morans_by_season.values, width, label='Miami', alpha=0.8)
axes[0].bar(x + width/2, phoenix_morans_by_season.values, width, label='Phoenix', alpha=0.8)
axes[0].set_xlabel('Season')
axes[0].set_ylabel("Moran's I")
axes[0].set_title("Spatial Autocorrelation by Season", fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(miami_morans_by_season.index)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# By year
miami_morans_by_year = miami_stats.groupby('year')['morans_i'].mean()
phoenix_morans_by_year = phoenix_stats.groupby('year')['morans_i'].mean()

axes[1].plot(miami_morans_by_year.index, miami_morans_by_year.values,
            marker='o', label='Miami', linewidth=2)
axes[1].plot(phoenix_morans_by_year.index, phoenix_morans_by_year.values,
            marker='s', label='Phoenix', linewidth=2)
axes[1].set_xlabel('Year')
axes[1].set_ylabel("Moran's I")
axes[1].set_title("Spatial Autocorrelation Trend (2016-2026)", fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./outputs/morans_i_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: morans_i_comparison.png")

### Hotspot Analysis

In [ ]:
# Plot: Hotspot percentage by season
fig, ax = plt.subplots(figsize=(12, 6))

miami_hotspots = miami_stats.groupby('season')['hotspot_pct'].mean()
phoenix_hotspots = phoenix_stats.groupby('season')['hotspot_pct'].mean()

x = np.arange(len(miami_hotspots))
width = 0.35

ax.bar(x - width/2, miami_hotspots.values, width, label='Miami', alpha=0.8)
ax.bar(x + width/2, phoenix_hotspots.values, width, label='Phoenix', alpha=0.8)

ax.set_xlabel('Season')
ax.set_ylabel('Hotspot Area (%)')
ax.set_title('High-Temperature Area (>75th percentile) by Season', fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(miami_hotspots.index)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('./outputs/hotspot_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: hotspot_analysis.png")

### Temperature Variability (Standard Deviation)

In [ ]:
# Plot: Temperature variability
fig, ax = plt.subplots(figsize=(12, 6))

miami_std = miami_stats.groupby('season')['std_temp'].mean()
phoenix_std = phoenix_stats.groupby('season')['std_temp'].mean()

x = np.arange(len(miami_std))
width = 0.35

ax.bar(x - width/2, miami_std.values, width, label='Miami', alpha=0.8, color='steelblue')
ax.bar(x + width/2, phoenix_std.values, width, label='Phoenix', alpha=0.8, color='coral')

ax.set_xlabel('Season')
ax.set_ylabel('Standard Deviation (°C)')
ax.set_title('LST Spatial Variability by Season', fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(miami_std.index)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('./outputs/temperature_variability.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: temperature_variability.png")

## Key Findings & Interpretation

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS: LST Comparison Miami vs Phoenix (2016-2026)")
print("="*70)

# Overall temperature
miami_mean = miami_stats['mean_temp'].mean()
phoenix_mean = phoenix_stats['mean_temp'].mean()
temp_diff = phoenix_mean - miami_mean

print(f"\n1. ABSOLUTE TEMPERATURE:")
print(f"   Miami mean LST:   {miami_mean:.2f}°C")
print(f"   Phoenix mean LST: {phoenix_mean:.2f}°C")
print(f"   Difference:       {temp_diff:.2f}°C (Phoenix is {temp_diff:.2f}°C hotter)")

# Seasonal patterns
print(f"\n2. SEASONAL PATTERNS:")
miami_seasonal = miami_stats.groupby('season')['mean_temp'].mean()
phoenix_seasonal = phoenix_stats.groupby('season')['mean_temp'].mean()

for season in ['DJF', 'MAM', 'JJA', 'SON']:
    m = miami_seasonal[season]
    p = phoenix_seasonal[season]
    print(f"   {season}: Miami={m:.1f}°C, Phoenix={p:.1f}°C (Δ={p-m:.1f}°C)")

# Spatial clustering
miami_morans_mean = miami_stats['morans_i'].mean()
phoenix_morans_mean = phoenix_stats['morans_i'].mean()

print(f"\n3. SPATIAL AUTOCORRELATION (Moran's I):")
print(f"   Miami:   {miami_morans_mean:.3f} {'(clustered)' if miami_morans_mean > 0.1 else '(random)'}")
print(f"   Phoenix: {phoenix_morans_mean:.3f} {'(clustered)' if phoenix_morans_mean > 0.1 else '(random)'}")
print(f"   Interpretation: {'Phoenix has stronger spatial clustering' if phoenix_morans_mean > miami_morans_mean else 'Miami has stronger spatial clustering'}")

# Variability
miami_std = miami_stats['std_temp'].mean()
phoenix_std = phoenix_stats['std_temp'].mean()

print(f"\n4. SPATIAL VARIABILITY (Std Dev):")
print(f"   Miami:   {miami_std:.2f}°C")
print(f"   Phoenix: {phoenix_std:.2f}°C")
print(f"   Interpretation: {'Phoenix has higher internal variability' if phoenix_std > miami_std else 'Miami has higher internal variability'}")

# Hotspots
miami_hotspot = miami_stats['hotspot_pct'].mean()
phoenix_hotspot = phoenix_stats['hotspot_pct'].mean()

print(f"\n5. HIGH-TEMPERATURE AREAS (% area >75th percentile):")
print(f"   Miami:   {miami_hotspot:.1f}%")
print(f"   Phoenix: {phoenix_hotspot:.1f}%")

# Trends
miami_trend = np.polyfit(miami_stats.groupby('year')['mean_temp'].mean().index,
                          miami_stats.groupby('year')['mean_temp'].mean().values, 1)[0]
phoenix_trend = np.polyfit(phoenix_stats.groupby('year')['mean_temp'].mean().index,
                           phoenix_stats.groupby('year')['mean_temp'].mean().values, 1)[0]

print(f"\n6. LONG-TERM TREND (2016-2026):")
print(f"   Miami:   {miami_trend:+.4f}°C/year")
print(f"   Phoenix: {phoenix_trend:+.4f}°C/year")
print(f"   Interpretation: {'Warming in both cities' if miami_trend > 0 and phoenix_trend > 0 else 'Cooling or mixed trends'}")

print("\n" + "="*70)

## Export Results

In [ ]:
# Save combined statistics
combined.to_csv('./outputs/LST_combined_statistics.csv', index=False)

# Create summary report
summary = f"""
LST ANALYSIS SUMMARY: Miami vs Phoenix (2016-2026)
={'='*60}

TEMPERATURE COMPARISON:
  Miami mean LST:   {miami_mean:.2f}°C
  Phoenix mean LST: {phoenix_mean:.2f}°C
  Difference:       {temp_diff:.2f}°C

SPATIAL AUTOCORRELATION (Moran's I):
  Miami:   {miami_morans_mean:.3f}
  Phoenix: {phoenix_morans_mean:.3f}

TEMPERATURE VARIABILITY (°C):
  Miami:   {miami_std:.2f}°C
  Phoenix: {phoenix_std:.2f}°C

TREND (°C/year):
  Miami:   {miami_trend:+.4f}°C/year
  Phoenix: {phoenix_trend:+.4f}°C/year

Data Source: MODIS/061/MOD11A1 (Daytime LST)
Temporal Granularity: Seasonal (DJF, MAM, JJA, SON)
Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d')}
"""

with open('./outputs/LST_analysis_summary.txt', 'w') as f:
    f.write(summary)

print("✓ Results exported:")
print("  - LST_combined_statistics.csv")
print("  - LST_analysis_summary.txt")
print("  - Individual city statistics (CSV files)")
print("  - Visualization outputs (PNG files)")

## Next Steps

1. **Add Heat Exposure Index**: Create parallel pipeline for HEI calculation
2. **Advanced Spatial Analysis**:
   - Local Indicators of Spatial Association (LISA)
   - Getis-Ord Gi* hotspot analysis
   - Spatial regression models
3. **Multi-city Comparison**: Extend to more cities across different climate zones
4. **Urban Heat Island (UHI) Analysis**: Compare urban vs rural pixels within each city
5. **Temporal Decomposition**: Separate trend, seasonal, and residual components
6. **Statistical Testing**: Mann-Kendall test for monotonic trends